In [5]:
pip install scipy


     ---------------------------------------- 0.0/60.8 kB ? eta -:--:--
     ------------ ------------------------- 20.5/60.8 kB 682.7 kB/s eta 0:00:01
     -------------------------------- ----- 51.2/60.8 kB 660.6 kB/s eta 0:00:01
     -------------------------------- ----- 51.2/60.8 kB 660.6 kB/s eta 0:00:01
     -------------------------------------- 60.8/60.8 kB 323.9 kB/s eta 0:00:00
   ---------------------------------------- 0.0/41.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/41.2 MB ? eta -:--:--
   ---------------------------------------- 0.1/41.2 MB 1.1 MB/s eta 0:00:37
   ---------------------------------------- 0.1/41.2 MB 1.0 MB/s eta 0:00:41
   ---------------------------------------- 0.2/41.2 MB 893.0 kB/s eta 0:00:46
   ---------------------------------------- 0.2/41.2 MB 981.9 kB/s eta 0:00:42
   ---------------------------------------- 0.3/41.2 MB 1.1 MB/s eta 0:00:39
   ---------------------------------------- 0.3/41.2 MB 999.9 kB/s eta 0:00:41


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Direktori dataset
train_dir = "seg_train/seg_train"
val_dir = "seg_test/seg_test"
# Augmentasi data
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, zoom_range=0.2, 
horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(train_dir, target_size=(150, 150), 
batch_size=32, class_mode='categorical')
val_generator = val_datagen.flow_from_directory(val_dir, target_size=(150, 150), batch_size=32, 
class_mode='categorical')
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
# Definisi model CNN
model = Sequential([
Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
MaxPooling2D(2,2),
Conv2D(64, (3,3), activation='relu'),
MaxPooling2D(2,2),
Flatten(),
Dense(128, activation='relu'),
Dropout(0.5),
Dense(6, activation='softmax')
])
# Kompilasi model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Training model
model.fit(train_generator, validation_data=val_generator, epochs=10)
# Simpan model
model.save('cnn_model.h5')

Found 14034 images belonging to 6 classes.
Found 3000 images belonging to 6 classes.


c:\Users\Uwiii\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Uwiii\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 610ms/step - accuracy: 0.4684 - loss: 1.4612

c:\Users\Uwiii\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


439/439 ━━━━━━━━━━━━━━━━━━━━ 302s 684ms/step - accuracy: 0.4686 - loss: 1.4605 - val_accuracy: 0.6080 - val_loss: 1.0861
Epoch 2/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 151s 344ms/step - accuracy: 0.6578 - loss: 0.9320 - val_accuracy: 0.7220 - val_loss: 0.7943
Epoch 3/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 158s 360ms/step - accuracy: 0.6865 - loss: 0.8511 - val_accuracy: 0.7353 - val_loss: 0.7412
Epoch 4/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 174s 395ms/step - accuracy: 0.7212 - loss: 0.7753 - val_accuracy: 0.7730 - val_loss: 0.6397
Epoch 5/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 223s 508ms/step - accuracy: 0.7312 - loss: 0.7536 - val_accuracy: 0.7910 - val_loss: 0.5902
Epoch 6/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 216s 492ms/step - accuracy: 0.7514 - loss: 0.6867 - val_accuracy: 0.8163 - val_loss: 0.5438
Epoch 7/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 216s 492ms/step - accuracy: 0.7582 - loss: 0.6684 - val_accuracy: 0.7857 - val_loss: 0.5839
Epoch 8/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 214s 488ms/step - accuracy: 0.7685 - loss: 0.66

In [15]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
# Load model yang telah dilatih
model = load_model('cnn_model.h5')
# Load label kelas
class_labels = list(train_generator.class_indices.keys())
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    # Mode Night Vision dengan konversi ke skala abu-abu
    night_vision = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    night_vision = cv2.applyColorMap(night_vision, cv2.COLORMAP_JET)
    # Preprocessing gambar
    img = cv2.resize(frame, (150, 150))
    img = img.astype("float32") / 255.0
    img = np.expand_dims(img, axis=0)
    # Prediksi kelas
    pred = model.predict(img)
    label = class_labels[np.argmax(pred)]
    # Tampilkan hasil
    cv2.putText(frame, f'Class: {label}', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow('Frame', frame)
    cv2.imshow('Night Vision', night_vision)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 374ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/st